In [1]:
import numpy as np
import pickle
resp1000 = pickle.load(open('resp1000_500.pkl', 'rb'))
f_gw = 0.01
theta_1 = 0
theta_2 = np.pi/6
theta_3 = np.pi/3
r = 1.46e11
c = 3e8

In [4]:
resp1000[1]/3600

np.float64(13.829546800493883)

In [51]:
def normalize_with_errors(z, rel_amp_err, phase_err):
    """
    Parameters
    ----------
    z : array_like (complex)
        Complex numbers z_k
    rel_amp_err : array_like (float)
        Relative amplitude errors (sigma_Ak / Ak)
    phase_err : array_like (float)
        Absolute phase errors (radians)

    Returns
    -------
    r : complex ndarray
        Normalized complex list z_k / z_0
    sigma_re : ndarray
        Error on real part of normalized response
    sigma_im : ndarray
        Error on imaginary part of normalized response
    sigma_A0 : float
        Absolute error on overall magnitude |z_0|
    """

    z = np.asarray(z)
    rel_amp_err = np.asarray(rel_amp_err)
    phase_err = np.asarray(phase_err)

    A = np.abs(z)
    phi = np.angle(z)

    # Normalization reference
    A0 = A[0]
    phi0 = phi[0]

    # Overall magnitude error
    sigma_A0 = rel_amp_err[0] * A0

    # Relative amplitudes and phase differences
    alpha = A / A0
    delta_phi = phi - phi0

    # Relative amplitude errors (ratio propagation)
    sigma_alpha = alpha * np.sqrt(rel_amp_err**2 + rel_amp_err[0]**2)

    # Phase difference errors
    sigma_delta_phi = np.sqrt(phase_err**2 + phase_err[0]**2)

    # Normalized complex response
    r = alpha * np.exp(1j * delta_phi)

    # Real and imaginary parts
    cos = np.cos(delta_phi)
    sin = np.sin(delta_phi)

    # Error propagation
    sigma_re = np.sqrt(
        (cos**2) * sigma_alpha**2 +
        (alpha**2) * (sin**2) * sigma_delta_phi**2
    )

    sigma_im = np.sqrt(
        (sin**2) * sigma_alpha**2 +
        (alpha**2) * (cos**2) * sigma_delta_phi**2
    )

    return r, A0, sigma_re, sigma_im, sigma_A0

In [69]:
def compute_fd_complex_response(f_gw,A, ra, dec, iota, psi):
    # GW propagation vector
    positions = np.array([[r*np.cos(theta_1),r*np.sin(theta_1),0],[r*np.cos(theta_2),r*np.sin(theta_3),0],[r*np.cos(theta_3),r*np.sin(theta_3),0],[r*np.cos(theta_1),r*np.sin(theta_1),0],[r*np.cos(theta_2),r*np.sin(theta_3),0],[r*np.cos(theta_3),r*np.sin(theta_3),0]])
    arms = np.array([[np.sin(theta_1),np.cos(theta_1),0],[np.sin(theta_2),np.cos(theta_2),0],[np.sin(theta_3),np.cos(theta_3),0],[-np.sin(theta_1),-np.cos(theta_1),0],[-np.sin(theta_2),-np.cos(theta_2),0],[-np.sin(theta_3),-np.cos(theta_3),0]])
    arm_lengths = [1e10,1e10,1e10,1e10,1e10,1e10]
    arm_lengths = np.array(arm_lengths)

    theta= np.pi/2- dec
    phi = ra
    k = -np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ])
    # 1. Amplitudes

    A_cross  = A*np.cos(iota)
    A_plus = A*(1+np.cos(iota)**2)/2
    print(np.array(A_cross)/A)
    print(np.array(A_plus)/A)
    def make_perp_basis(k):
        k = k / np.linalg.norm(k)
        if np.abs(k[2]) < 0.99:
            tmp = np.array([0,0,1])
        else:
            tmp = np.array([1,0,0])
        ex = np.cross(tmp, k)
        ex /= np.linalg.norm(ex)
        ey = np.cross(k, ex)
        return ex, ey

    ex, ey = make_perp_basis(k)

    # Polarization tensors
    e_plus = np.outer(ex, ex) - np.outer(ey, ey)
    e_cross = np.outer(ex, ey) + np.outer(ey, ex)

    # Rotate by polarization angle psi
    e_plus_rot  = e_plus * np.cos(2 * psi) + e_cross * np.sin(2 * psi)
    e_cross_rot = -e_plus * np.sin(2 * psi) + e_cross * np.cos(2 * psi)

    # --- CLOCK ANTENNA GEOMETRY ---
    # n̂ = photon propagation direction
    nk = np.einsum("di,i->d", arms, k)
    denom = 1.0 - nk

    # Geometry-only detector tensor
    D = 0.5 * np.einsum("di,dj->dij", arms, arms) / denom[:, None, None]

    # Antenna factors
    F_plus  = np.einsum("dij,ij->d", D, e_plus_rot)
    F_cross = np.einsum("dij,ij->d", D, e_cross_rot)
    print('F_plus:')
    print(F_plus)
    print('F_cross:')
    print(F_cross)

    # Keep original phase convention
    tau = np.dot(positions, k) / c
   
    phase_delay = 2 * np.pi * f_gw * arm_lengths * denom / c
    transfer = 1.0 - np.exp(-1j * phase_delay)
   
    # Pure antenna-pattern response (no transfer function)
    S = 2/np.pi*(F_plus * A_plus + 1j*F_cross * A_cross) * np.exp(-1j * 2 * np.pi * f_gw * tau) * transfer
    
    S_model = S/S[0]
    print('model')
    print(S_model)
    
    A_model = np.abs(S[0])
    
    return S_model, A_model



In [70]:
class GWNetworkLikelihood():
    def __init__(self, data_S,data_A, error_S_re,error_S_im,error_A, model):
        # We only accept Complex Strain (S) data, NO time delay (tau) data
        self.data_S = data_S     # Complex array [Amp * exp(i*Phase)]
        self.error_S_re = error_S_re   # Error array
        self.error_S_im = error_S_im   # Error array
        self.error_A = error_A   # Error array
        self.data_A = data_A
        self.model = model 
        


    def log_likelihood(self,parameters):
    
        print(parameters)
        S_model, A_model = self.model(f_gw, **parameters)
     
        print('data')
        print(self.data_S)
        delta_S = self.data_S - S_model
        
        

        # Split into Real and Imaginary parts
        delta_S_re = np.real(delta_S)
        delta_S_im = np.imag(delta_S)
        delta_A = self.data_A-A_model
        
        sigma_S_re = self.error_S_re**2
        sigma_S_im = self.error_S_im**2
        sigma_A = self.error_A**2
        
        # Calculate chi-squared for Real and Imaginary parts
        ln_L_S_re = -0.5 * np.sum((delta_S_re**2 / sigma_S_re) + np.log(2 * np.pi * sigma_S_re))
        ln_L_S_im = -0.5 * np.sum((delta_S_im**2 / sigma_S_im) + np.log(2 * np.pi * sigma_S_im))
        ln_L_A = -0.5 * np.sum((delta_A**2 / sigma_A) + np.log(2 * np.pi * sigma_A))
        
        # Total Likelihood is just the Strain part
        return ln_L_S_re + ln_L_S_im + ln_L_A

In [71]:
S = resp1000[0]
amp_error = resp1000[2]
phase_error = resp1000[3]
true_S = resp1000[-1]

injection_parameters = dict(
    ra=0.2,                 # Right Ascension (rad)
    dec=np.pi/2-0.8,                # Declination (rad)
    psi=np.pi/3,               # Polarization angle (rad)
    A  = 5e-20,
    iota = np.pi/3
)

S_true, A_true = compute_fd_complex_response(f_gw, **injection_parameters)


0.5000000000000001
0.6250000000000001
F_plus:
[-0.10316012  0.14384585  0.15814484 -0.13745125  0.40408764  0.83064145]
F_cross:
[-0.4161459  -0.21960732  0.02388478 -0.55447563 -0.61691458  0.12545264]
model
[ 1.        +0.j          0.02694381+0.69919511j  0.35727093+0.32405523j
  1.07004652+0.32923015j -0.8394615 +0.59245275j -0.47197555+0.70439684j]


In [64]:
S

[np.complex128(-1.1508127656383322e-20+5.3651455137650935e-21j),
 np.complex128(-3.951476658489268e-21-8.073568097490956e-21j),
 np.complex128(-5.853305640720126e-21-1.8282854294427964e-21j),
 np.complex128(-1.396284722213036e-20+1.6725431586278066e-21j),
 np.complex128(6.7616795054288624e-21-1.1183219242804721e-20j),
 np.complex128(1.902959426560834e-21-1.0774009692892056e-20j)]

In [65]:
S_norm, A0, sigma_re, sigma_im, sigma_A0 = normalize_with_errors(S,amp_error,phase_error)

In [66]:
likelihood = GWNetworkLikelihood(
    data_S=S_norm, 
    data_A = A0,
    error_S_re=sigma_re,
    error_S_im=sigma_im,
    error_A =sigma_A0,
    model=compute_fd_complex_response
)


In [67]:
injection_parameters = dict(
    ra=0.2,                 # Right Ascension (rad)
    dec=np.pi/2-0.8,                # Declination (rad)
    psi=np.pi/3,               # Polarization angle (rad)
    A  = 3.5e-20,
    iota = np.pi/3
)

random_parameters = dict(
    ra=56,                 # Right Ascension (rad)
    dec=2346,                # Declination (rad)
    psi=np.pi/8,               # Polarization angle (rad)
    A  = 10e-20,
    iota = np.pi/7
)

In [68]:
log_l_true = likelihood.log_likelihood(parameters = injection_parameters)

print(f"Log Likelihood at Injection: {log_l_true}")

# 2. Test the likelihood at a RANDOM point

log_l_random = likelihood.log_likelihood(parameters = random_parameters)

print(f"Log Likelihood at Random Point: {log_l_random}")

{'ra': 0.2, 'dec': 0.7707963267948965, 'psi': 1.0471975511965976, 'A': 3.5e-20, 'iota': 1.0471975511965976}
0.5000000000000001
0.625
model
[ 1.        +0.j          0.02694381+0.69919511j  0.35727093+0.32405523j
  1.07004652+0.32923015j -0.8394615 +0.59245275j -0.47197555+0.70439684j]
data
[ 1.        +0.j          0.01338672+0.70779453j  0.35697142+0.32529089j
  1.05233708+0.34526888j -0.85480921+0.57325081j -0.49437257+0.70572983j]
Log Likelihood at Injection: 7.156030938666625
{'ra': 56, 'dec': 2346, 'psi': 0.39269908169872414, 'A': 1e-19, 'iota': 0.4487989505128276}
0.9009688679024191
0.9058724504646833
model
[ 1.        -0.j          0.74200971+1.17501925j -1.25989413+0.56922545j
  0.95574973+0.95387594j  0.7071904 +1.21938673j -0.43533386+0.95596528j]
data
[ 1.        +0.j          0.01338672+0.70779453j  0.35697142+0.32529089j
  1.05233708+0.34526888j -0.85480921+0.57325081j -0.49437257+0.70572983j]
Log Likelihood at Random Point: -92954.63656795246


In [60]:
A0

np.float64(1.2697314225437142e-20)

In [61]:
sigma_re

array([0.01330625, 0.01069698, 0.00952385, 0.02021634, 0.01328715,
       0.01085368])

In [39]:
S_norm, A0, sigma_re, sigma_im, sigma_A0 = normalize_with_errors(S,amp_error,phase_error)

In [40]:
S_true

array([ 1.        +0.j        ,  0.69833379-0.04392822j,
        0.35727093+0.32405523j,  1.07004652+0.32923015j,
        0.61267899+0.82481532j, -0.47197555+0.70439684j])

In [41]:
true_S_norm = true_S/true_S[0]

In [16]:
delta_S = (true_S_norm-S_norm)

In [17]:
delta_S_re = np.real(delta_S)
delta_S_im = np.imag(delta_S)

In [18]:
delta_S_re

array([ 0.        ,  0.01455388, -0.00016169,  0.01842498,  0.01216192,
        0.01879511])

In [19]:
sigma_re

array([0.01330625, 0.01069698, 0.00952385, 0.02021634, 0.01328715,
       0.01085368])

In [20]:
-0.5*(np.sum((delta_S_re/sigma_re)**2)+np.sum((delta_S_im/sigma_im)**2)+np.sum(np.log(2 * np.pi * sigma_re))+np.sum(np.log(2 * np.pi * sigma_re)))

np.float64(9.77233909991034)

In [21]:
np.sum(np.log(2 * np.pi * sigma_re))
       

np.float64(-15.229482337184592)

In [22]:
delta_S_im

array([-0.        , -0.00992816, -0.00321515, -0.01432008,  0.01796984,
       -0.00157978])

In [23]:
sigma_im

array([0.01074735, 0.01043241, 0.00927986, 0.0123068 , 0.01271381,
       0.01166404])

In [24]:
-2*np.sum(np.log(2 * np.pi * sigma_re**2))

np.float64(82.97245414565052)

In [25]:
S_true

array([ 1.        +0.j        ,  0.02694381+0.69919511j,
        0.35727093+0.32405523j,  1.07004652+0.32923015j,
       -0.8394615 +0.59245275j, -0.47197555+0.70439684j])

In [26]:
true_S_norm

array([ 1.        -0.j        ,  0.02794059+0.69786637j,
        0.35680973+0.32207573j,  1.07076206+0.3309488j ,
       -0.84264729+0.59122065j, -0.47557746+0.70415004j])

In [27]:
S_true-true_S_norm

array([ 0.        +0.j        , -0.00099679+0.00132874j,
        0.0004612 +0.0019795j , -0.00071553-0.00171865j,
        0.0031858 +0.0012321j ,  0.00360191+0.0002468j ])

In [33]:
np.sum((delta_S_im/sigma_im)**2)

np.float64(4.395725307291629)

In [34]:
np.sum((delta_S_re/sigma_re)**2)

np.float64(6.518561167256874)

In [35]:
A0

np.float64(1.2697314225437142e-20)

In [38]:
true_A = np.abs(true_S)

In [39]:
A0-true_A

array([-2.35660070e-22,  3.66459548e-21,  6.48079035e-21, -1.79718936e-21,
       -6.15462067e-22,  1.70808136e-21])